# Tableau - Driver Lap Times - Data Validation and Sanity Checks

A copy of KPI 3, preparing an extended dataset (all GPs in 2015-2019) for feature engineering.

In [25]:
import pandas as pd

# read csv file
df_laptimes = pd.read_csv('/Users/frankdong/Documents/Analytics Local/williams-racing-strategies/processed_data/tableau-driver-lap-times.csv')

# dataframe basic info
print(df_laptimes.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11228 entries, 0 to 11227
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   race_id                11228 non-null  int64 
 1   gp_year                11228 non-null  int64 
 2   gp_name                11228 non-null  object
 3   gp_round               11228 non-null  int64 
 4   driver_id              11228 non-null  int64 
 5   driver_name            11228 non-null  object
 6   rookie_or_experienced  11228 non-null  object
 7   lap_number             11228 non-null  int64 
 8   lap_time               11228 non-null  object
 9   lap_time_ms            11228 non-null  int64 
dtypes: int64(6), object(4)
memory usage: 877.3+ KB
None


## Column data types

In [26]:
print(df_laptimes.dtypes)

race_id                   int64
gp_year                   int64
gp_name                  object
gp_round                  int64
driver_id                 int64
driver_name              object
rookie_or_experienced    object
lap_number                int64
lap_time                 object
lap_time_ms               int64
dtype: object


## Missing or null values

In [27]:
df_laptimes.isnull().sum() # No nulls present across the dataset!

race_id                  0
gp_year                  0
gp_name                  0
gp_round                 0
driver_id                0
driver_name              0
rookie_or_experienced    0
lap_number               0
lap_time                 0
lap_time_ms              0
dtype: int64

## Check for duplicates

In [28]:
df_laptimes.duplicated().sum() # No duplicates found

np.int64(0)

## Summary statistics

In [29]:
df_laptimes.describe()

,race_id,gp_year,gp_round,driver_id,lap_number,lap_time_ms
count,11228.000000,11228.000000,11228.000000,11228.000000,11228.000000,1.122800e+04
mean,979.059672,2017.036961,10.791414,510.156840,30.179551,9.700934e+04
std,30.368401,1.416217,5.843842,402.966136,18.168466,4.668000e+04
min,926.000000,2015.000000,1.000000,9.000000,1.000000,6.841900e+04
25%,953.000000,2016.000000,6.000000,13.000000,15.000000,8.353250e+04
50%,980.000000,2017.000000,11.000000,822.000000,29.000000,9.533950e+04
75%,1006.000000,2018.000000,16.000000,840.000000,44.000000,1.045998e+05
max,1030.000000,2019.000000,21.000000,847.000000,78.000000,2.118323e+06


## Filtering valid lap times

In [30]:
# Compute the median lap time for each driver’s race session (defined by gp_year, gp_round and driver_id)
df_laptimes['median_lap_time_ms'] = df_laptimes.groupby(['gp_year', 'gp_round', 'driver_id'])['lap_time_ms'].transform('median')

MAX_FILTER_PCT = 1.05 # multiplier for the maximum lap time for filtering purposes

# Compute the max_lap_time = MAX_FILTER_PCT * median lap time - any times landing outside this is excluded. 
df_laptimes['max_lap_time_ms'] = df_laptimes['median_lap_time_ms'] * MAX_FILTER_PCT

# Keep only laps below the max filter - dynamic filter based on 107% rule
df_laptimes = df_laptimes[df_laptimes['lap_time_ms'] <= df_laptimes['max_lap_time_ms']]

# drop both extra median and max columns
#df_laptimes.drop(columns=['median_lap_time_ms', 'max_lap_time_ms'], inplace=True)

# Reset index for df_laptimes
df_laptimes = df_laptimes.reset_index(drop=True)

# Re-check the dataset after filtering
print(df_laptimes.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9856 entries, 0 to 9855
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   race_id                9856 non-null   int64  
 1   gp_year                9856 non-null   int64  
 2   gp_name                9856 non-null   object 
 3   gp_round               9856 non-null   int64  
 4   driver_id              9856 non-null   int64  
 5   driver_name            9856 non-null   object 
 6   rookie_or_experienced  9856 non-null   object 
 7   lap_number             9856 non-null   int64  
 8   lap_time               9856 non-null   object 
 9   lap_time_ms            9856 non-null   int64  
 10  median_lap_time_ms     9856 non-null   float64
 11  max_lap_time_ms        9856 non-null   float64
dtypes: float64(2), int64(6), object(4)
memory usage: 924.1+ KB
None


In [31]:
df_laptimes.describe()

,race_id,gp_year,gp_round,driver_id,lap_number,lap_time_ms,median_lap_time_ms,max_lap_time_ms
count,9856.000000,9856.000000,9856.000000,9856.000000,9856.000000,9856.000000,9856.000000,9856.000000
mean,979.336546,2017.049209,10.815239,511.949472,31.358056,92361.412946,92762.929028,97401.075479
std,30.475416,1.419625,5.868719,402.661841,17.744999,12328.826790,12499.754308,13124.742023
min,926.000000,2015.000000,1.000000,9.000000,1.000000,68419.000000,70258.500000,73771.425000
25%,953.000000,2016.000000,6.000000,13.000000,16.000000,82489.250000,83407.000000,87577.350000
50%,981.000000,2017.000000,11.000000,822.000000,31.000000,92162.000000,92263.000000,96876.150000
75%,1006.000000,2018.000000,16.000000,840.000000,45.000000,102127.500000,102297.000000,107411.850000
max,1030.000000,2019.000000,21.000000,847.000000,78.000000,154823.000000,154823.000000,162564.150000


In [32]:
# Export the validated lap times dataframe to a new CSV file, called "driver-lap-times-validated.csv"
df_laptimes.to_csv('/Users/frankdong/Documents/Analytics Local/williams-racing-strategies/processed_data/tableau-driver-lap-times-dynamic-validated.csv')